In [ ]:
#~
using Images

# Comm Link Modeling in 42

Eric Stoneking, Mar 2026

Although modeling communication link performance is not among 42's original motivations, it has been noticed that the geometrical inputs to comm link modeling are readily obtained from 42's orbit and attitude outputs.  So we added some code and input/output to facilitate comm link modeling.

A comm link is composed of a transmitter (Tx), a receiver (Rx), and the line of sight (Path) that connects them.  Input power to the Tx antenna is amplified by the Tx antenna gain, propagates through space (and perhaps an atmosphere) with attending losses, and is amplified further by the Rx antenna gain.  The result of this chain is the carrier signal at the Rx antenna output.  Comm link performance is the ratio of this carrier to the noise in the receiver itself.  Several other outputs of interest are also generated.

Anything upstream of the Tx antenna input or downstream of the Rx antenna output is beyond the scope of this modeling effort.  

# Inputs

Comm link input parameters are defined in Inp_CommLink.txt.  Some, we hope, are self-explanatory.  We give some description of the less obvious entries.

## Adjust Positions for Delay

Due to the finite speed of light, the position of Tx at the current time is not exactly where the comm signal being received at Rx at the current time was transmitted.  For those scenarios where this error in path length and direction is  objectionable, 42 can back-propagate the Tx position to reduce it.  Delay Accuracy is used as a stopping condition for the iteration.

## Link Type

A comm link may be an UPLINK, DOWNLINK, or CROSSLINK.  For an uplink, Tx is a ground station and Rx is a spacecraft.  For a downlink, Tx is a spacecraft and Rx is a ground station.  For a crosslink, both Tx and Rx are spacecraft.

## Frequency

The link carrier frequency is needed to calculate the freespace path loss and to scale the Doppler shift.

## Link Noise Floor

Link calculations are most conveniently expressed in decibels (dB).  A zero-power signal, however, would thus have a value of negative infinity dBW, which is awkward.  We address this by assigning a user-provided Link Noise Floor.  When a link is occulted, the carrier will be set to this value rather than to negative infinity.

## Tx and Rx Terminal ID, Body, and Mounting Angles

These lines are interpreted differently depending on whether the Tx or Rx terminal are mounted on a ground station or a spacecraft.  In the case of a ground station, the Terminal ID refers to the ground station ID (see Inp_Sim.txt for the list of ground stations), and the Body and Mounting Angle inputs are ignored.  For a spacecraft, the Terminal ID is the Spacecraft index as defined in Inp_Sim.txt, the Body is the S/C body in which the terminal is mounted, and the Mounting Angle inputs provide the orientation of the antenna with respect to the body.

## Antenna Peak Gain, Floor Gain, and Mesh File

The antenna pattern is represented as a triangular mesh stored as a *Wavefront obj* file.  In the Model folder, see Ant*.obj for some notional examples.  Figure #ref{Ant_Xband} shows a notional parabolic antenna pattern.  

The antenna gain is found by finding the intersection of the mesh with a ray projected from the origin in the direction of the link path.  The mesh construction must follow some rules so that it is correctly interpreted. 

The mesh must be a closed volume with the origin in its interior.  The mesh faces must all be triangles.  Each face's vertices must be listed in counter-clockwise order as viewed from the outside.  A ray projected from the origin must intersect the mesh at exactly one point.

It is natural to map the radial extent of the mesh in a chosen direction to the antenna gain expressed in dB.  Since gains expressed in dB are often negative, and mesh radii aren't, some scaling and offset is needed.

From Inp_CommLink.txt, the user provides the antenna's desired Peak Gain and Floor Gain (which may be negative).  On loading, the mesh file's maximum and minimum radii are found.  During the simulation, the antenna gain in the desired direction, $G$, is found from the mesh radius in that direction, $r$, by:

\begin{equation}
G = G_{\rm floor} + \frac{(G_{\rm peak}-G_{\rm floor})}{(r_{\rm max}-r_{\rm min})}(r-r_{\rm min})
\end{equation}

In constructing the mesh for the antenna pattern, one should use the inverse to assign $r$ as a function of $G$ for a given direction:

\begin{equation}
r = r_{\rm min} + \frac{(r_{\rm max}-r_{\rm min})}{(G_{\rm peak}-G_{\rm floor})}(G-G_{\rm floor})
\end{equation}

As a convenience, the file Model/Ant_Isotropic.obj is a unit sphere triangular mesh.  To construct a custom antenna pattern, make a copy of it and scale each vertex as desired.  The function in the appendix was used to construct the Ant_Xband.obj antenna pattern.  See also Utilities/SculptAntPatt.ipynb.

In the degenerate case where the mesh minimum and maximum radii are equal, the antenna gain will be reported as the Floor Gain.

## Rx System Noise Power

The total noise power of the Rx system is needed to compute the carrier-to-noise ratio.  None of the inputs to this item depend on the link geometry, so rather than make the user provide any subcomponents of system noise and just parrot back their sum, we depend on the user to provide the total itself.

## Atmophere Loss Mean and Random Walk

We provide a simple stochastic model of atmospheric losses.  The user provides the mean loss, as well as the standard deviation and correlation time for a random walk process.  This allows a model that is more interesting than just a simple constant, but is far less involved than a physics-based model like the ITU-R model.


In [ ]:
function SculptAntPatt()

      infile = open("../Model/Ant_Isotropic.obj");
      outfile = open("../Model/Ant_XBand.obj","w");

      PeakGain = 40.0;
      FloorGain = -50.0;
      MaxRad = 1.0;
      MinRad = 0.01;

      while !eof(infile) 
         line = readline(infile);
         if startswith(line,"v")
            cols = split(line," ");
            x = parse(Float64,cols[2]);
            y = parse(Float64,cols[3]);
            z = parse(Float64,cols[4]);
            r = sqrt(x^2+y^2+z^2);
            DirVec = [x/r,y/r,z/r];
            ang = acos(z/r);
            G = FloorGain + (PeakGain-FloorGain)*(abs(sinc(5*ang/pi)));
            r = MinRad + (MaxRad-MinRad)/(PeakGain-FloorGain)*(G-FloorGain);
            write(outfile,"v $(r*DirVec[1]) $(r*DirVec[2]) $(r*DirVec[3])\n");
         else
            write(outfile,line*"\n");
         end
      end;

      close(infile);
      close(outfile);
end;

In [ ]:
#~ caption="Notional Antenna Pattern Mesh"
#~ ref="Ant_Xband"
load("Ant_Xband.png")

# Outputs

Each link produces its own output file, named Link*.csv (where * is the link number).  It is a comma-delimited text file, with labels in the first row.  The outputs are:


| Name   | Units | Description |
|:-------|:-----|:------------------|
| Time | sec | Sim Time |
| Doppler | Hz | See Doppler note below |
| Loss | dB | Total link loss |
| Delay | sec | Time-of-flight delay |
| Carrier | dBW | Carrier signal amplitude at Rx antenna output |
| Noise | dBW | Rx system noise power provided by the user |
| CNR | dB | Carrier-to-noise ratio |
| EIRP | dBW | See EIRP note below |
| FSPL | dB | Free Space Path Loss |
| PwrFluxDensity | dBW | Power flux density received at Rx antenna input |
| Range | m | Distance between Tx and Rx |
| RangeRate | m/s | Rate of change of Range |
| TxAntGain | dB | Tx gain derived from antenna pattern and path direction |
| RxAntGain | dB | Rx gain derived from antenna pattern and path direction |
| TxOcculted | 0/1 | 1 if Tx end of path is occulted by its World |
| RxOcculted | 0/1 | 1 if Rx end of path is occulted by its World |
| PathIsOcculted | 0/1 | TxOcculted OR RxOcculted |

### Doppler

This is the frequency shift of the carrier frequency expressed in Hz.  For example, if Tx and Rx are moving toward each other at $0.01c$ and the carrier frequency is $3.0$ GHz, the tabulated Doppler value would be 3.0E7 Hz, and the observed carrier frequency would be shifted to 3.03 GHz.

### EIRP

This is the effective isotropic radiated power.  This is a measure of the power output by the Tx antenna in the direction of Rx.  It is the product of the Tx input power and the Tx antenna gain.


# Acknowledgements

We'd like to acknowledge and thank Haisam Ido, who saw a gap that 42 could fill.  We'd like to thank the NOS3 project for funding this work, and Michael Gross for his expert review and suggestions from an RF comms perspective.

# References

[1] Louis J. Ippolito, Jr.  __Satellite Communications Systems Engineering: Atmospheric Effects, Satellite Link Design and System Performance__.  Wiley, 2008.

[2] James R. Wertz, David F. Everett, Jeffery J. Puschell.  __Space Mission Engineering: The New SMAD__.  Microcosm Press, 2011.